Pre Proces

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# # TotalCharges を数値化（EDAでやった前処理を再掲）
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

# ① 目的変数 y を分離
y = (df["Churn"] == "Yes").astype(int)

# ② 特徴量 X（不要な列を落とす）
X = df.drop(columns=["customerID", "Churn"])

print("X の列数（エンコード前）:", X.shape[1])
print("カテゴリ列の数:", X.select_dtypes("str").shape[1])
print("y の解約率:", y.mean().round(3))

X の列数（エンコード前）: 19
カテゴリ列の数: 15
y の解約率: 0.265


In [2]:
# ③ カテゴリ変数をワンホットエンコード
cat_cols = X.select_dtypes("str").columns   # "object" → "str" に変更（警告対策）
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print("エンコード後の列数:", X.shape[1])
print("エンコード後の列の型:")
print(X.dtypes.value_counts())
X.head(3)

エンコード後の列数: 30
エンコード後の列の型:
bool       26
int64       2
float64     2
Name: count, dtype: int64


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True


In [3]:
# ④ 数値列だけスケールを揃える（正規化）
scaler = StandardScaler()
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]  # SeniorCitizenは既に0/1なので除外
X[num_cols] = scaler.fit_transform(X[num_cols])

# 訓練用とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("訓練データ:", X_train.shape)
print("テストデータ:", X_test.shape)
print("訓練の解約率:", y_train.mean().round(3))
print("テストの解約率:", y_test.mean().round(3))

訓練データ: (5634, 30)
テストデータ: (1409, 30)
訓練の解約率: 0.265
テストの解約率: 0.265


### 最初のモデルはロジスティック回帰

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# モデルを作って学習
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# テストデータで予測
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))   # ここだけ修正
print("\n=== classification report ===")
print(classification_report(y_test, y_pred, digits=3))

Accuracy: 0.806

=== classification report ===
              precision    recall  f1-score   support

           0      0.849     0.896     0.872      1035
           1      0.659     0.559     0.605       374

    accuracy                          0.806      1409
   macro avg      0.754     0.727     0.738      1409
weighted avg      0.799     0.806     0.801      1409



In [5]:
# class_weight="balanced" を追加するだけ
model_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
model_balanced.fit(X_train, y_train)

y_pred_b = model_balanced.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred_b), 3))
print("\n=== classification report (balanced) ===")
print(classification_report(y_test, y_pred_b, digits=3))

Accuracy: 0.739

=== classification report (balanced) ===
              precision    recall  f1-score   support

           0      0.902     0.723     0.803      1035
           1      0.505     0.783     0.614       374

    accuracy                          0.739      1409
   macro avg      0.704     0.753     0.708      1409
weighted avg      0.797     0.739     0.753      1409



In [6]:
import pandas as pd

# balancedモデルの係数を取り出す
coef = pd.DataFrame({
    "feature": X_train.columns,
    "coef": model_balanced.coef_[0]
}).sort_values("coef", ascending=False)

print("=== 解約を強める要因 トップ8 ===")
print(coef.head(8).to_string(index=False))
print("\n=== 解約を抑える要因 トップ8 ===")
print(coef.tail(8).to_string(index=False))

=== 解約を強める要因 トップ8 ===
                       feature     coef
   InternetService_Fiber optic 1.218764
                  TotalCharges 0.483092
           StreamingMovies_Yes 0.420979
PaymentMethod_Electronic check 0.401275
               StreamingTV_Yes 0.395463
          PaperlessBilling_Yes 0.337768
             MultipleLines_Yes 0.325431
                 SeniorCitizen 0.152062

=== 解約を抑える要因 トップ8 ===
           feature      coef
    Dependents_Yes -0.231312
   TechSupport_Yes -0.268014
OnlineSecurity_Yes -0.333978
  PhoneService_Yes -0.345657
    MonthlyCharges -0.544348
 Contract_One year -0.718914
            tenure -1.155041
 Contract_Two year -1.415403


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 不均衡対策: 多数派/少数派 の比率を重みとして渡す
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight =", round(ratio, 2))

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=ratio,   # ロジ回帰の class_weight="balanced" に相当
    eval_metric="logloss",
    random_state=42,
)
xgb.fit(X_train, y_train)

y_pred_x = xgb.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred_x), 3))
print("\n=== classification report (XGBoost) ===")
print(classification_report(y_test, y_pred_x, digits=3))

scale_pos_weight = 2.77
Accuracy: 0.758

=== classification report (XGBoost) ===
              precision    recall  f1-score   support

           0      0.906     0.748     0.819      1035
           1      0.530     0.786     0.633       374

    accuracy                          0.758      1409
   macro avg      0.718     0.767     0.726      1409
weighted avg      0.806     0.758     0.770      1409



In [9]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_
}).sort_values("importance", ascending=False)

print("=== XGBoost 特徴量重要度 トップ10 ===")
print(importance.head(10).to_string(index=False))

=== XGBoost 特徴量重要度 トップ10 ===
                       feature  importance
             Contract_Two year    0.361332
   InternetService_Fiber optic    0.166607
             Contract_One year    0.147251
            InternetService_No    0.084518
           StreamingMovies_Yes    0.044357
PaymentMethod_Electronic check    0.026909
                        tenure    0.020420
            OnlineSecurity_Yes    0.014230
          PaperlessBilling_Yes    0.013122
             MultipleLines_Yes    0.011539


In [10]:
# 3モデルの結果を表にまとめる（解約者=ラベル1 の指標を中心に）
results = pd.DataFrame({
    "model": ["LogReg (素)", "LogReg (balanced)", "XGBoost (balanced)"],
    "accuracy": [0.806, 0.739, 0.758],
    "recall (解約)": [0.559, 0.783, 0.786],
    "precision (解約)": [0.659, 0.505, 0.530],
    "f1 (解約)": [0.605, 0.614, 0.633],
})
print(results.to_string(index=False))

             model  accuracy  recall (解約)  precision (解約)  f1 (解約)
        LogReg (素)     0.806        0.559           0.659    0.605
 LogReg (balanced)     0.739        0.783           0.505    0.614
XGBoost (balanced)     0.758        0.786           0.530    0.633


## モデル比較と結論

### 指標の選択
本問題は解約予測であり、ビジネス上の目的は「離反しそうな顧客を事前に捕捉し引き留める」こと。
そのため、解約者を取りこぼさない **recall（再現率）** を主要指標とし、
空振り（precision低下）とのバランスを **f1-score** で評価した。
データは解約26.5%の不均衡データであり、accuracy単独では「全員継続」予測（73.5%）と
大差ない値になるため、評価指標として不適切と判断した。

### 結果
- 素のロジスティック回帰は accuracy 0.806 だが解約recallが0.559と低く、解約者の44%を見逃す。
- `class_weight="balanced"` で不均衡に対処すると、recallが0.783へ大幅改善（accuracyは0.739へ低下）。
- XGBoostはrecall 0.786を保ちつつf1が最高（0.633）。本データでの最良モデル。
- ただしXGBoostの優位は僅差であり、解釈性に優れるロジスティック回帰も実用上有力。

### 特徴量の知見（2モデルで一致）
線形モデルの係数と木系モデルの重要度の双方で、
**契約タイプ（長期契約ほど引き留め）** と **光回線（最大の解約リスク）** が一貫して上位。
異なる原理のモデルが同じ結論に至ったことで、知見の信頼性が高い。

### 注意点（相関の影響）
tenure（継続月数）はロジ回帰では強い引き留め要因だが、XGBoostでは重要度が低い。
これはtenureとContractが強く相関するため、木系モデルでContractに重要度が吸収されたもの。
単一モデルの重要度を鵜呑みにせず、特徴量間の相関構造を踏まえて解釈する必要がある。

### EDAの仮説との照合
EDAでは「月額が高いほど解約」と見えたが、多変数モデルでは月額単独の係数はマイナスに転じた。
これは「月額の高さ」の実体が光回線契約であったことを示し、
単変数の相関と多変数での寄与が異なる典型例である。